In [0]:
orders_silver = spark.table("workspace.default.orders_silver")
customers_silver = spark.table("workspace.default.customers_silver")
products_silver = spark.table("workspace.default.products_silver")
sellers_silver = spark.table("workspace.default.sellers_silver")
order_items_silver = spark.table("workspace.default.order_items_silver")
payments_silver = spark.table("workspace.default.payments_silver")
reviews_silver = spark.table("workspace.default.reviews_silver")
geolocation_silver = spark.table("workspace.default.geolocation_silver")
category_translation_silver = spark.table(
    "workspace.default.category_translation_silver"
)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
def check_nulls(df, table_name, columns):
    print(f"\n--- NULL CHECK: {table_name} ---")

    for column_name in columns:
        null_count = (
            df.filter(col(column_name).isNull())
              .count()
        )

        print(f"{column_name}: {null_count} NULLs")

In [0]:
check_nulls(
    orders_silver,
    "orders_silver",
    [
        "order_id",
        "customer_id",
        "order_status",
        "purchase_timestamp"
    ]
)

In [0]:
check_nulls(
    customers_silver,
    "customers_silver",
    [
        "customer_id",
        "customer_unique_id"
    ]
)

In [0]:
check_nulls(
    products_silver,
    "products_silver",
    [
        "product_id"
    ]
)

In [0]:
check_nulls(
    sellers_silver,
    "sellers_silver",
    [
        "seller_id"
    ]
)

In [0]:
check_nulls(
    order_items_silver,
    "order_items_silver",
    [
        "order_id",
        "item_id",
        "product_id",
        "seller_id"
    ]
)

In [0]:
check_nulls(
    payments_silver,
    "payments_silver",
    [
        "order_id",
        "payment_type",
        "payment_amount"
    ]
)

In [0]:
def check_unique(df, table_name, column_name):
    total_rows = df.count()

    unique_rows = (
        df.select(column_name)
          .distinct()
          .count()
    )

    duplicates = total_rows - unique_rows

    print(f"\n--- UNIQUENESS CHECK: {table_name} ---")
    print(f"Column: {column_name}")
    print(f"Total rows: {total_rows}")
    print(f"Unique values: {unique_rows}")
    print(f"Duplicate records: {duplicates}")

In [0]:
check_unique(
    orders_silver,
    "orders_silver",
    "order_id"
)

In [0]:
check_unique(
    customers_silver,
    "customers_silver",
    "customer_id"
)

In [0]:
check_unique(
    products_silver,
    "products_silver",
    "product_id"
)

In [0]:
check_unique(
    sellers_silver,
    "sellers_silver",
    "seller_id"
)

In [0]:
check_unique(
    order_items_silver,
    "order_items_silver",
    "order_id"
)

In [0]:
def check_composite_unique(df, table_name, columns):
    total_rows = df.count()

    unique_rows = (
        df.select(columns)
          .distinct()
          .count()
    )

    duplicates = total_rows - unique_rows

    print(f"\n--- COMPOSITE UNIQUENESS: {table_name} ---")
    print(f"Columns: {columns}")
    print(f"Duplicate records: {duplicates}")

In [0]:
check_composite_unique(
    order_items_silver,
    "order_items_silver",
    ["order_id", "item_id"]
)

In [0]:
invalid_products = products_silver.filter(
    (col("weight_g") < 0) |
    (col("length_cm") < 0) |
    (col("height_cm") < 0) |
    (col("width_cm") < 0) |
    (col("photos_qty") < 0)
)

print("Invalid product records:", invalid_products.count())

In [0]:
invalid_payments = payments_silver.filter(
    (col("payment_amount") <= 0) |
    (col("payment_installments") < 1)
)

print("Invalid payment records:", invalid_payments.count())

In [0]:
invalid_items = order_items_silver.filter(
    (col("price") < 0) |
    (col("freight_value") < 0)
)

print("Invalid order item records:", invalid_items.count())

In [0]:
invalid_order_dates = orders_silver.filter(
    (col("approved_timestamp") < col("purchase_timestamp")) |
    (col("delivered_carrier_timestamp") < col("purchase_timestamp")) |
    (col("delivered_customer_timestamp") < col("purchase_timestamp"))
)

print("Orders with invalid date sequence:", invalid_order_dates.count())

In [0]:
orders_without_customer = (
    orders_silver
    .join(
        customers_silver,
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Orders with missing customers:",
    orders_without_customer.count()
)

In [0]:
items_without_order = (
    order_items_silver
    .join(
        orders_silver,
        on="order_id",
        how="left_anti"
    )
)

print(
    "Items with missing orders:",
    items_without_order.count()
)

In [0]:
items_without_product = (
    order_items_silver
    .join(
        products_silver,
        on="product_id",
        how="left_anti"
    )
)

print(
    "Items with missing products:",
    items_without_product.count()
)

In [0]:
items_without_seller = (
    order_items_silver
    .join(
        sellers_silver,
        on="seller_id",
        how="left_anti"
    )
)

print(
    "Items with missing sellers:",
    items_without_seller.count()
)

In [0]:
invalid_order_dates.select(
    "order_id",
    "order_status",
    "purchase_timestamp",
    "approved_timestamp",
    "delivered_carrier_timestamp",
    "delivered_customer_timestamp",
    "estimated_delivery_timestamp"
).show(30, truncate=False)

In [0]:
print(
    "Approved before purchase:",
    orders_silver.filter(
        col("approved_timestamp") < col("purchase_timestamp")
    ).count()
)

print(
    "Carrier delivery before purchase:",
    orders_silver.filter(
        col("delivered_carrier_timestamp") < col("purchase_timestamp")
    ).count()
)

print(
    "Customer delivery before purchase:",
    orders_silver.filter(
        col("delivered_customer_timestamp") < col("purchase_timestamp")
    ).count()
)

In [0]:
invalid_payments.show(20, truncate=False)

In [0]:
invalid_payments.select(
    "order_id",
    "payment_type",
    "payment_installments",
    "payment_amount"
).show(20, truncate=False)

In [0]:
print("Orders without customers:",
      orders_silver.join(
          customers_silver,
          "customer_id",
          "left_anti"
      ).count())

print("Items without orders:",
      order_items_silver.join(
          orders_silver,
          "order_id",
          "left_anti"
      ).count())

print("Items without products:",
      order_items_silver.join(
          products_silver,
          "product_id",
          "left_anti"
      ).count())

print("Items without sellers:",
      order_items_silver.join(
          sellers_silver,
          "seller_id",
          "left_anti"
      ).count())

In [0]:
print(
    "Invalid products:",
    products_silver.filter(
        (col("weight_g") < 0) |
        (col("length_cm") < 0) |
        (col("height_cm") < 0) |
        (col("width_cm") < 0) |
        (col("photos_qty") < 0)
    ).count()
)

In [0]:
print(
    "Invalid order items:",
    order_items_silver.filter(
        (col("price") < 0) |
        (col("freight_value") < 0)
    ).count()
)

In [0]:
reviews_silver.printSchema()

In [0]:
payments_silver.groupBy(
    "payment_valid"
).count().show()